# First capacity-model experiment

This notebook is the interactive laboratory for the first model application.

We have one shared capacity pool:

1. user demand arrives stochastically according to a Poisson model;
2. user demand is served first;
3. scheduled work receives the remaining capacity; and
4. unfinished work carries forward as backlog.

The notebook changes assumptions and displays results. The reusable model logic remains in src/capacity_model.py.

## What we are trying to learn

For each candidate capacity, we estimate:

- expected total cost;
- user-demand fill rate;
- user backlog; and
- scheduled-work backlog.

The selected capacity is the one with the lowest estimated total cost under the assumptions below.

## Project environment

The next cell keeps the notebook isolated from the global Python installation. It creates or checks the project-local .venv, installs the notebook dependencies there if needed, and registers the Python (capacity-model) kernel.

If this notebook was opened with a different kernel, run the cell once, select Python (capacity-model), and then run the notebook from the beginning.

In [1]:
from pathlib import Path
import subprocess
import sys

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

venv_dir = project_dir / ".venv"
venv_python = venv_dir / "bin" / "python"
requirements = project_dir / "requirements-notebook.txt"

if not venv_python.exists():
    print("Creating the project-local virtual environment...")
    subprocess.check_call([sys.executable, "-m", "venv", str(venv_dir)])

dependencies_ready = subprocess.run(
    [str(venv_python), "-c", "import ipykernel, jupyterlab, matplotlib"],
    capture_output=True,
).returncode == 0

if not dependencies_ready:
    print("Installing notebook dependencies into .venv...")
    subprocess.check_call(
        [str(venv_python), "-m", "pip", "install", "-r", str(requirements)]
    )

subprocess.check_call(
    [
        str(venv_python),
        "-m",
        "ipykernel",
        "install",
        "--sys-prefix",
        "--name",
        "capacity-model-venv",
        "--display-name",
        "Python (capacity-model)",
    ],
    stdout=subprocess.DEVNULL,
)

print(f"Project environment: {venv_dir}")
print(f"Current notebook interpreter: {sys.executable}")
if Path(sys.executable).resolve() == venv_python.resolve():
    print("Ready: this notebook is running inside Python (capacity-model).")
else:
    print("Setup complete. Select the Python (capacity-model) kernel, then rerun from the beginning.")

Project environment: /Users/panos/.codex/.chatgpt-projects/g-p-6a63465261a081919fbf5287435162a5/capacity-model-starter/.venv
Current notebook interpreter: /Users/panos/Documents/project optimus/capacity problem v1.0/.venv/bin/python
Setup complete. Select the Python (capacity-model) kernel, then rerun from the beginning.


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "matplotlib",
])

In [2]:
from pathlib import Path
import sys

# This works when the notebook is opened from the project directory or its notebooks folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir / "src"))

from capacity_model import (
    choose_capacity,
    evaluate_capacities,
    generate_poisson_scenarios,
    poisson_quantile,
    simulate_scenario,
)

In [3]:
# Main experiment controls. Change these values and rerun the cells below.
lambda_rate = 8.0              # expected user arrivals per period
horizon = 24                    # number of periods in each scenario
scenario_count = 2_000         # possible demand futures to simulate
scheduled_per_period = 4       # scheduled work released each period
capacity_max = 20               # largest candidate capacity to test

capacity_cost = 1.0            # cost per capacity unit per period
user_backlog_cost = 10.0        # penalty per user-backlog unit per period
scheduled_backlog_cost = 1.0    # penalty per scheduled-backlog unit per period
confidence = 0.95               # one-period demand protection level
seed = 20260830                 # keeps the experiment reproducible

user_reserve = poisson_quantile(lambda_rate, confidence)
print(f"{confidence:.0%} one-period user-demand reserve: {user_reserve}")

95% one-period user-demand reserve: 13


## 1. Inspect one possible future

The Poisson model does not produce one guaranteed demand path. It produces many possible paths. We first inspect one of them so that the backlog mechanics are visible.

In [4]:
demand_scenarios = generate_poisson_scenarios(
    lambda_rate,
    horizon,
    scenario_count,
    seed=seed,
)
inspection_capacity = 14
all_arrivals = [arrival for path in demand_scenarios for arrival in path]
overload_draws = sum(arrival > inspection_capacity for arrival in all_arrivals)
overload_scenarios = sum(
    any(arrival > inspection_capacity for arrival in path)
    for path in demand_scenarios
)
print(f"Maximum generated arrival: {max(all_arrivals)}")
print(f"Periods above capacity {inspection_capacity}: {overload_draws}")
print(f"Scenarios containing an overload: {overload_scenarios}/{len(demand_scenarios)}")

# Pick an overload-containing path so the backlog mechanism is visible.
first_demand_path = next(
    (path for path in demand_scenarios if max(path) > inspection_capacity),
    demand_scenarios[0],
)

one_result = simulate_scenario(
    capacity=inspection_capacity,
    user_arrivals=first_demand_path,
    scheduled_arrivals=scheduled_per_period,
    capacity_cost_per_unit=capacity_cost,
    user_backlog_cost_per_unit=user_backlog_cost,
    scheduled_backlog_cost_per_unit=scheduled_backlog_cost,
)

print("period | user arrivals | user served | scheduled served | user backlog | scheduled backlog")
for row in one_result.periods:
    print(
        f"{row.period:6d} | {row.user_arrivals:13d} | {row.user_served:11d} | "
        f"{row.scheduled_served:16d} | {row.user_backlog_end:12d} | "
        f"{row.scheduled_backlog_end:17d}"
    )

Maximum generated arrival: 21
Periods above capacity 14: 814
Scenarios containing an overload: 674/2000
period | user arrivals | user served | scheduled served | user backlog | scheduled backlog
     0 |             8 |           8 |                4 |            0 |                 0
     1 |             8 |           8 |                4 |            0 |                 0
     2 |             4 |           4 |                4 |            0 |                 0
     3 |             9 |           9 |                4 |            0 |                 0
     4 |            11 |          11 |                3 |            0 |                 1
     5 |             9 |           9 |                5 |            0 |                 0
     6 |            12 |          12 |                2 |            0 |                 2
     7 |            11 |          11 |                3 |            0 |                 3
     8 |            14 |          14 |                0 |            0 |     

The priority rule is visible here: scheduled work is processed only after current user work and user backlog have used the available capacity.

## 2. Compare candidate capacities

Every candidate sees exactly the same simulated futures. This is important: otherwise a capacity could appear better simply because it received easier random demand.

In [5]:
results = evaluate_capacities(
    capacities=range(capacity_max + 1),
    demand_scenarios=demand_scenarios,
    scheduled_arrivals=scheduled_per_period,
    capacity_cost_per_unit=capacity_cost,
    user_backlog_cost_per_unit=user_backlog_cost,
    scheduled_backlog_cost_per_unit=scheduled_backlog_cost,
)

print("capacity | expected cost | user fill rate | user backlog cost | scheduled backlog cost")
for result in results:
    print(
        f"{result.capacity:8d} | {result.total_cost:14.2f} | "
        f"{result.user_fill_rate:14.2%} | {result.user_backlog_cost:17.2f} | "
        f"{result.scheduled_backlog_cost:23.2f}"
    )

selected = choose_capacity(results)
print(f"\nSelected capacity: {selected.capacity}")
print(f"Lowest estimated total cost: {selected.total_cost:.2f}")

capacity | expected cost | user fill rate | user backlog cost | scheduled backlog cost
       0 |       25131.94 |          0.00% |          23931.94 |                 1200.00
       1 |       22155.94 |          1.14% |          20931.94 |                 1200.00
       2 |       19180.59 |          2.64% |          17932.66 |                 1199.93
       3 |       16207.28 |          4.67% |          14935.66 |                 1199.63
       4 |       13241.83 |          7.61% |          11947.37 |                 1198.46
       5 |       10300.47 |         12.21% |           8985.86 |                 1194.61
       6 |        7417.18 |         20.37% |           6088.80 |                 1184.39
       7 |        4733.84 |         36.48% |           3413.64 |                 1152.20
       8 |        2681.92 |         60.94% |           1439.09 |                 1050.83
       9 |        1565.87 |         81.33% |            501.43 |                  848.44
      10 |        1021.

## 3. Optional visual comparison

If Matplotlib is installed, this cell plots the two main trade-offs. The notebook remains usable without it; the table above is the authoritative result.

In [6]:
try:
    import matplotlib.pyplot as plt

    capacities = [result.capacity for result in results]
    costs = [result.total_cost for result in results]
    fill_rates = [result.user_fill_rate for result in results]

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(capacities, costs, marker="o")
    axes[0].axvline(selected.capacity, color="tab:red", linestyle="--", label="selected")
    axes[0].set(title="Expected total cost", xlabel="Capacity", ylabel="Cost")
    axes[0].legend()

    axes[1].plot(capacities, fill_rates, marker="o", color="tab:green")
    axes[1].axvline(selected.capacity, color="tab:red", linestyle="--", label="selected")
    axes[1].set(title="User-demand fill rate", xlabel="Capacity", ylabel="Fill rate")
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()

    figure.tight_layout()
    plt.show()
except ImportError:
    print("Matplotlib is not installed; use the result table above.")

Matplotlib is not installed; use the result table above.


## 4. Sensitivity to the demand rate

The selected capacity is not a universal answer. It depends on the expected demand rate and the relative costs. This small experiment repeats the comparison for several Poisson rates.

To keep this interactive cell quick, it uses a smaller scenario sample than the main experiment. Increase sensitivity_scenario_count when you want a more precise final comparison.

In [7]:
sensitivity_scenario_count = min(scenario_count, 500)
print(f"Using {sensitivity_scenario_count} scenarios per demand rate for this interactive check.")
print("lambda | 95% reserve | selected capacity | expected cost | user fill rate")
for rate in (4.0, 8.0, 12.0):
    scenarios = generate_poisson_scenarios(
        rate,
        horizon,
        sensitivity_scenario_count,
        seed=seed,
    )
    rate_results = evaluate_capacities(
        range(capacity_max + 1),
        scenarios,
        scheduled_arrivals=scheduled_per_period,
        capacity_cost_per_unit=capacity_cost,
        user_backlog_cost_per_unit=user_backlog_cost,
        scheduled_backlog_cost_per_unit=scheduled_backlog_cost,
    )
    rate_selected = choose_capacity(rate_results)
    print(
        f"{rate:6.1f} | {poisson_quantile(rate, confidence):11d} | "
        f"{rate_selected.capacity:17d} | {rate_selected.total_cost:14.2f} | "
        f"{rate_selected.user_fill_rate:14.2%}"
    )

Using 500 scenarios per demand rate for this interactive check.
lambda | 95% reserve | selected capacity | expected cost | user fill rate
   4.0 |           8 |                 9 |         241.05 |         99.74%
   8.0 |          13 |                14 |         361.03 |         99.63%
  12.0 |          18 |                19 |         480.40 |         99.64%


## Interpretation

This first notebook does not optimize individual scheduled jobs. It answers the foundational question:

> How much total capacity should we provide when user demand is uncertain, user demand is protected, and scheduled work fills the residual capacity?

The next modeling layer can replace the filler assumption with prescheduled jobs, deadlines, importance weights, and dependencies while reusing the same scenario and backlog framework.